# HyperKT Objective 01 — reproducible Colab implementation
This notebook implements the described hypergraph, causal transformer, Beta heads and uncertainty evaluation. **No manuscript performance values are assumed.** Upload a legally obtained CSV with `user_id,skill_id,correct` and optionally `timestamp`. For EdNet/Junyi, prepare these columns from the source files first. Run on a Colab GPU. Student-level split; train-only co-occurrence. A smoke test uses synthetic data and is **not publishable evidence**.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

In [ ]:
import os,random,copy,json,time
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,accuracy_score,f1_score,brier_score_loss,log_loss
SEED=42
def seed_all(s):
 random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)
seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
# Set to a CSV in Colab Drive or /content. Empty path runs synthetic smoke test only.
CSV_PATH=''
MAX_LEN=200; BATCH_SIZE=32; EPOCHS=60; PATIENCE=8
D=256; HEADS=8; LAYERS=4; HYPEREDGES=32


In [ ]:
if CSV_PATH:
 df=pd.read_csv(CSV_PATH,encoding='utf-8',low_memory=False)
else:
 rng=np.random.default_rng(SEED); rows=[]
 for u in range(120):
  skills=rng.integers(0,30,size=35)
  for t,s in enumerate(skills):rows.append((u,int(s),int(rng.random()<(.35+.3*(s%4)/3)),t))
 df=pd.DataFrame(rows,columns=['user_id','skill_id','correct','timestamp'])
 print('SYNTHETIC SMOKE TEST ONLY: results cannot support manuscript claims')
required={'user_id','skill_id','correct'}
assert required.issubset(df.columns),f'Missing {required-set(df.columns)}'
df=df.dropna(subset=list(required)).copy()
df['correct']=pd.to_numeric(df['correct'],errors='coerce')
df=df[df['correct'].isin([0,1])].copy()
if 'timestamp' not in df:df['timestamp']=np.arange(len(df))
df['_order']=np.arange(len(df))
df=df.sort_values(['user_id','timestamp','_order'],kind='stable')
# ID encoding uses global vocabulary without outcome-derived statistics.
df['skill'],skill_names=pd.factorize(df['skill_id'],sort=True)
df['skill']=df['skill']+1 # zero reserved for padding
N_SKILLS=int(df.skill.max());print('rows',len(df),'students',df.user_id.nunique(),'skills',N_SKILLS)
users=df.user_id.drop_duplicates().to_numpy()
train_users,tmp=train_test_split(users,test_size=.30,random_state=SEED)
val_users,test_users=train_test_split(tmp,test_size=.50,random_state=SEED)
assert not (set(train_users)&set(val_users) or set(train_users)&set(test_users) or set(val_users)&set(test_users))
def make_sequences(ids):
 out=[]
 for _,g in df[df.user_id.isin(ids)].groupby('user_id',sort=False):
  s=g.skill.to_numpy(dtype=np.int64);r=g.correct.to_numpy(dtype=np.int64)
  # non-overlapping targets: each interaction from index 1 onwards predicted exactly once
  for end in range(2,len(s)+1,MAX_LEN):
   stop=min(end+MAX_LEN-1,len(s)+1)
   # segment includes prior interaction and up to MAX_LEN-1 targets
   start=end-2
   ss=s[start:stop];rr=r[start:stop]
   if len(ss)>1:out.append((ss,rr))
 return out
train_seq=make_sequences(train_users);val_seq=make_sequences(val_users);test_seq=make_sequences(test_users)
print('segments:',len(train_seq),len(val_seq),len(test_seq))
class KTData(Dataset):
 def __init__(self,seq):self.seq=seq
 def __len__(self):return len(self.seq)
 def __getitem__(self,i):
  s,r=self.seq[i];return torch.tensor(s[:-1]),torch.tensor(r[:-1]),torch.tensor(s[1:]),torch.tensor(r[1:],dtype=torch.float32)
def collate(batch):
 B=len(batch);T=max(len(z[0]) for z in batch)
 a=torch.zeros(B,T,dtype=torch.long);b=a.clone();q=a.clone();y=torch.zeros(B,T);mask=torch.zeros(B,T,dtype=torch.bool)
 for i,(s,r,t,v) in enumerate(batch):
  n=len(s);a[i,:n]=s;b[i,:n]=r;q[i,:n]=t;y[i,:n]=v;mask[i,:n]=True
 return a,b,q,y,mask
def loader(seq,shuffle=False):return DataLoader(KTData(seq),batch_size=BATCH_SIZE,shuffle=shuffle,collate_fn=collate)


In [ ]:
# Train-only co-occurrence: fraction of student windows containing both skills.
# Compute on complete training histories, not segmented targets.
G=np.zeros((N_SKILLS,N_SKILLS),dtype=np.float32);windows=0
for _,g in df[df.user_id.isin(train_users)].groupby('user_id',sort=False):
 s=g.skill.to_numpy()-1
 for i in range(len(s)):
  unique=np.unique(s[i:i+5]);G[np.ix_(unique,unique)]+=1;windows+=1
G/=max(windows,1)
G=torch.tensor(G,device=DEVICE)
print('Co-occurrence windows',windows)


In [ ]:
class HyperKT(nn.Module):
 def __init__(self,n,d=D,heads=HEADS,layers=LAYERS,m=HYPEREDGES,drop=.1,forget=True,hyper=True):
  super().__init__();self.n=n;self.d=d;self.forget=forget;self.hyper=hyper
  self.skill=nn.Embedding(n+1,d//2,padding_idx=0);self.response=nn.Embedding(2,d//2)
  self.proj=nn.Linear(d,d);self.incidence=nn.Parameter(torch.randn(n,m)*.02)
  self.strength=nn.Parameter(torch.zeros(m));self.edge=nn.Parameter(torch.randn(m,d)*.02)
  self.decay_raw=nn.Parameter(torch.tensor(-2.9706)) # softplus approx 0.05
  enc=nn.TransformerEncoderLayer(d,heads,dim_feedforward=4*d,dropout=drop,batch_first=True,norm_first=True)
  self.encoder=nn.TransformerEncoder(enc,layers,enable_nested_tensor=False)
  self.gate=nn.Linear(2*d,d);self.query=nn.Embedding(n+1,d,padding_idx=0)
  self.alpha=nn.Sequential(nn.Linear(2*d,d),nn.ReLU(),nn.Dropout(drop),nn.Linear(d,1))
  self.beta=nn.Sequential(nn.Linear(2*d,d),nn.ReLU(),nn.Dropout(drop),nn.Linear(d,1))
 def incidence_matrix(self):return F.softmax(self.incidence,dim=-1)
 def hg_loss(self,target):
  H=self.incidence_matrix();w=F.softplus(self.strength)
  pred=torch.sigmoid((H*w)@H.T)
  # entropy sign convention: positive entropy encourages sharp membership when minimized
  entropy=-(H*torch.log(H.clamp_min(1e-8))).sum(-1).mean()
  return F.mse_loss(pred,target)+.01*w.abs().sum()+.05*entropy
 def forward(self,s,r,q,mask):
  B,T=s.shape
  z=self.proj(torch.cat([self.skill(s),self.response(r)],-1))
  pos=torch.arange(T,device=s.device).float();freq=torch.exp(torch.arange(0,self.d,2,device=s.device).float()*(-np.log(10000)/self.d))
  pe=torch.zeros(T,self.d,device=s.device);pe[:,0::2]=torch.sin(pos[:,None]*freq);pe[:,1::2]=torch.cos(pos[:,None]*freq)
  if self.forget:
   # recent positions get stronger positional magnitude
   pe=pe*torch.exp(-F.softplus(self.decay_raw)*(T-1-pos)[:,None]/MAX_LEN)
  z=z+pe[None]
  causal=torch.triu(torch.ones(T,T,device=s.device,dtype=torch.bool),diagonal=1)
  h=self.encoder(z,mask=causal,src_key_padding_mask=~mask)
  if self.hyper:
   H=self.incidence_matrix();structure=F.embedding(s,H)@self.edge
   gate=torch.sigmoid(self.gate(torch.cat([h,structure],-1)))
   h=gate*h+(1-gate)*structure
  # Target-skill query is known before its response; avoids leakage from r_(t+1).
  state=torch.cat([h,self.query(q)],-1)
  a=F.softplus(self.alpha(state)).squeeze(-1)+1
  b=F.softplus(self.beta(state)).squeeze(-1)+1
  return a/(a+b),a,b
model=HyperKT(N_SKILLS).to(DEVICE)
print('parameters:',sum(p.numel() for p in model.parameters()))


In [ ]:
def soft_ece(p,y,bins=10):
 # Differentiable soft-bin approximation (training only); report hard ECE separately.
 centers=torch.linspace(.05,.95,bins,device=p.device)
 weights=(1-(p[:,None]-centers[None,:]).abs()/.1).clamp_min(0)
 counts=weights.sum(0).clamp_min(1e-8)
 return ((weights*(p-y)[:,None]).sum(0).abs()/counts*counts/counts.sum()).sum()
def epoch(model,dl,opt=None):
 training=opt is not None;model.train(training);losses=[]
 for s,r,q,y,mask in dl:
  s,r,q,y,mask=[x.to(DEVICE) for x in (s,r,q,y,mask)]
  with torch.set_grad_enabled(training):
   p,a,b=model(s,r,q,mask);pred=p[mask];target=y[mask]
   loss=F.binary_cross_entropy(pred,target)+.01*soft_ece(pred,target)
   if model.hyper:loss=loss+.1*model.hg_loss(G)
   if training:
    opt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
  losses.append(float(loss.detach()))
 return float(np.mean(losses)) if losses else float('nan')
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-5)
best=float('inf');bad=0;history=[];best_state=None
for e in range(EPOCHS):
 tr=epoch(model,loader(train_seq,True),optimizer);va=epoch(model,loader(val_seq))
 history.append({'epoch':e+1,'train_loss':tr,'val_loss':va})
 print(f'Epoch {e+1:02d} train={tr:.4f} val={va:.4f}')
 if va<best-1e-5:best=va;bad=0;best_state=copy.deepcopy(model.state_dict())
 else:
  bad+=1
  if bad>=PATIENCE:break
model.load_state_dict(best_state);torch.save(model.state_dict(),'hyperkt_best.pt')
pd.DataFrame(history).to_csv('training_history.csv',index=False)


In [ ]:
@torch.no_grad()
def predictions(model,seq,mc=1):
 model.train(mc>1) # activate dropout for MC; no gradient updates
 ys=[];ps=[];ep=[];ale=[]
 for s,r,q,y,mask in loader(seq):
  s,r,q,y,mask=[x.to(DEVICE) for x in (s,r,q,y,mask)]
  passes=[];vars=[]
  for _ in range(mc):
   p,a,b=model(s,r,q,mask);passes.append(p[mask]);vars.append((a*b/((a+b)**2*(a+b+1)))[mask])
  pp=torch.stack(passes);ys.extend(y[mask].cpu().numpy());ps.extend(pp.mean(0).cpu().numpy());ep.extend(pp.var(0,unbiased=False).cpu().numpy());ale.extend(torch.stack(vars).mean(0).cpu().numpy())
 model.eval();return np.array(ys),np.array(ps),np.array(ep),np.array(ale)
def ece(y,p,bins=10):
 ix=np.minimum((p*bins).astype(int),bins-1);return sum((ix==i).mean()*abs(y[ix==i].mean()-p[ix==i].mean()) for i in range(bins) if (ix==i).any())
def metrics(y,p):
 return {'auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,'accuracy':float(accuracy_score(y,p>=.5)),'f1':float(f1_score(y,p>=.5,zero_division=0)),'ece':float(ece(y,p)),'brier':float(brier_score_loss(y,p)),'nll':float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),'n':len(y)}
y,p,ep,ale=predictions(model,test_seq,mc=30)
result=metrics(y,p);print(json.dumps(result,indent=2))
pd.DataFrame({'correct':y,'probability':p,'epistemic_variance':ep,'beta_variance':ale}).to_csv('test_predictions.csv',index=False)
with open('test_metrics.json','w') as f:json.dump(result,f,indent=2)
H=model.incidence_matrix().detach().cpu().numpy();pd.DataFrame(H,index=skill_names).to_csv('learned_hyperedges.csv')


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots();ax.plot([0,1],[0,1],linestyle='--')
for i in range(10):
 m=(p>=i/10)&(p<((i+1)/10 if i<9 else 1.000001))
 if m.any():ax.scatter(p[m].mean(),y[m].mean(),s=max(12,int(m.sum()/max(len(y),1)*500)))
ax.set(xlabel='Mean predicted probability',ylabel='Observed correctness',title='Test reliability diagram');fig.savefig('reliability.png',dpi=180);plt.show()
print('Saved: hyperkt_best.pt, training_history.csv, test_predictions.csv, test_metrics.json, learned_hyperedges.csv, reliability.png')


## Publication checklist
Run the **real datasets** separately; repeat five seeds; implement and evaluate BKT, DKT, SAKT, GNNKT and DTransformer on the **identical splits**; run ablations (no hypergraph, no forgetting, no Beta head), bootstrap confidence intervals and expert evaluation of learned skill groups. Record data versions and preprocessing counts. The paper currently contains incompatible dataset counts and unsupported results; update only after verification. The Beta head here models a predictive probability distribution, not a formally inferred Bayesian posterior. Co-occurrence is undirected association, not evidence of directed prerequisite causality. The manuscript must reflect these distinctions.